# FAISS Vector Search Experiments

Standalone notebook for the SIFT, GIST, and GloVe FAISS experiments. It embeds the helper code that was previously imported from `src/*.py`, so it can run end-to-end without local module imports.

The notebook writes CSV tables to `results/tables/` and plots to `results/plots/`.


## Run Notes

Run the notebook from top to bottom. SIFT and GIST use Euclidean search (`METRIC_L2`), while GloVe uses L2-normalized vectors with inner-product search (`METRIC_INNER_PRODUCT`). GloVe is capped by `GLOVE_MAX_VECTORS` for a repeatable, feasible run.


In [7]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from __future__ import annotations

from pathlib import Path
import os
import time

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Detect project root: one level up from notebooks/, or cwd if already at root
_nb_dir = Path.cwd()
PROJECT_ROOT = (_nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir).resolve()

RAW_DATA_ROOT = PROJECT_ROOT / 'data' / 'raw'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'
PLOTS_DIR = PROJECT_ROOT / 'results' / 'plots'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_DATA_ROOT:', RAW_DATA_ROOT)
print('Exists:', RAW_DATA_ROOT.exists())

NOISY_QUERIES_DIR = PROJECT_ROOT / 'data' / 'noisy_queries'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
NOISY_QUERIES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
ARTIFACTS: list[str] = []


PROJECT_ROOT: C:\Users\vaspn\Downloads\Vector-Search-FAISS
RAW_DATA_ROOT: C:\Users\vaspn\Downloads\Vector-Search-FAISS\data\raw
Exists: True


## Standalone Helpers

This cell contains the dataset readers, preprocessing, metrics, index helpers, benchmarking code, Relative Contrast, and query-noise utilities used by the experiments.


In [9]:
# From src/data_loading.py
def _resolve_path(path: str | Path) -> Path:
    """Resolve a path relative to project root when needed."""
    p = Path(path)
    return p if p.is_absolute() else (PROJECT_ROOT / p)


def read_fvecs(path: str | Path) -> np.ndarray:
    """Read a `.fvecs` file into a float32 array of shape (n, d)."""
    file_path = _resolve_path(path)
    # Read everything as int32 first because the metadata and dimension markers are integers
    raw = np.fromfile(file_path, dtype=np.int32)

    if raw.size == 0:
        return np.empty((0, 0), dtype=np.float32)

    # In fvecs format, the first 4 bytes of every vector store its dimensionality (d)
    dim = int(raw[0])
    if dim <= 0:
        raise ValueError(f"Invalid vector dimension in {file_path}: {dim}")
    # Each vector row takes up (dim + 1) elements: 1 element for dimension + d elements for data
    if raw.size % (dim + 1) != 0:
        raise ValueError(f"Corrupted fvecs file: {file_path}")

    # Reshape into a 2D grid where each row is [dim_marker, v_1, v_2, ..., v_d]
    vectors_i32 = raw.reshape(-1, dim + 1)
    # Make sure every single row starts with the correct dimension value
    if not np.all(vectors_i32[:, 0] == dim):
        raise ValueError(f"Inconsistent vector dimensions in {file_path}")

    # Slice off the first column (dimension marker), then safely view/cast the remaining bits as float32
    return vectors_i32[:, 1:].view(np.float32).copy()


def read_ivecs(path: str | Path) -> np.ndarray:
    """Read an `.ivecs` file into an int32 array of shape (n, d)."""
    file_path = _resolve_path(path)
    raw = np.fromfile(file_path, dtype=np.int32)

    if raw.size == 0:
        return np.empty((0, 0), dtype=np.int32)

    # ivecs follows the exact same structure as fvecs, but the vector data itself is also integers (IDs)
    dim = int(raw[0])
    if dim <= 0:
        raise ValueError(f"Invalid vector dimension in {file_path}: {dim}")
    if raw.size % (dim + 1) != 0:
        raise ValueError(f"Corrupted ivecs file: {file_path}")

    vectors_i32 = raw.reshape(-1, dim + 1)
    if not np.all(vectors_i32[:, 0] == dim):
        raise ValueError(f"Inconsistent vector dimensions in {file_path}")

    # Slice off the first column; no .view() needed here since the data is natively int32 indices
    return vectors_i32[:, 1:].copy()


def load_sift() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Load the SIFT dataset as (xb, xq, xlearn, gt)."""
    root = RAW_DATA_ROOT / "sift"
    xb = read_fvecs(root / "sift_base.fvecs") # Base database vectors
    xq = read_fvecs(root / "sift_query.fvecs") # Query target vectors
    xlearn = read_fvecs(root / "sift_learn.fvecs") # Hyperparameter/quantizer training vectors
    gt = read_ivecs(root / "sift_groundtruth.ivecs") # Ground truth index maps
    return xb, xq, xlearn, gt


def load_gist() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Load the GIST dataset as (xb, xq, xlearn, gt)."""
    root = RAW_DATA_ROOT / "gist"
    xb = read_fvecs(root / "gist_base.fvecs")
    xq = read_fvecs(root / "gist_query.fvecs")
    xlearn = read_fvecs(root / "gist_learn.fvecs")
    gt = read_ivecs(root / "gist_groundtruth.ivecs")
    return xb, xq, xlearn, gt


# From src/preprocessing.py
def l2_normalize(x: np.ndarray) -> np.ndarray:
    """L2-normalize a 2D float32 array of vectors and return it."""
    if x.ndim != 2:
        raise ValueError(f"`x` must be 2D, got shape {x.shape}.")

    # Ensure data is float32 without duplicating memory if already correct
    x_f32 = x.astype(np.float32, copy=False)

    # Scales every row vector to have a length of 1.0.
    # This transforms an Inner Product search into Cosine Similarity.
    faiss.normalize_L2(x_f32)
    return x_f32


# From src/metrics.py
def recall_at_k(I_approx: np.ndarray, I_gt: np.ndarray, k: int) -> float:
    """Compute average Recall@k between approximate and exact neighbors."""
    # Input Validation
    if k <= 0:
        raise ValueError("`k` must be positive.")
    if I_approx.ndim != 2 or I_gt.ndim != 2:
        raise ValueError("`I_approx` and `I_gt` must be 2D arrays.")
    if I_approx.shape[0] != I_gt.shape[0]:
        raise ValueError(
            "`I_approx` and `I_gt` must have the same number of queries. "
            f"Got {I_approx.shape[0]} and {I_gt.shape[0]}."
        )
    if I_approx.shape[1] < k or I_gt.shape[1] < k:
        raise ValueError(
            "`k` cannot exceed the second dimension of inputs. "
            f"Got k={k}, I_approx.shape[1]={I_approx.shape[1]}, I_gt.shape[1]={I_gt.shape[1]}."
        )

    # Calculate recall
    hits = 0
    # Loop through each query row
    for i in range(I_gt.shape[0]):
        # Count how many of the top-k predicted IDs match the true top-k ground truth IDs
        hits += len(set(I_approx[i, :k]).intersection(set(I_gt[i, :k])))
    # Return the total matching hits divided by the maximum possible hits
    return hits / (I_gt.shape[0] * k)


# From src/benchmarks.py
import time



def build_ground_truth(
    xb: np.ndarray, xq: np.ndarray, k: int, metric: int
) -> tuple[np.ndarray, np.ndarray]:
    """Build exact k-NN ground truth with a Flat index."""
    # Input Validation
    if k <= 0:
        raise ValueError("`k` must be positive.")
    if xb.ndim != 2 or xq.ndim != 2:
        raise ValueError("`xb` and `xq` must be 2D arrays.")
    if xb.shape[1] != xq.shape[1]:
        raise ValueError(
            "`xb` and `xq` must have the same dimensionality. "
            f"Got {xb.shape[1]} and {xq.shape[1]}."
        )

    # Force float32 type without copying if already float32
    xb_f32 = xb.astype(np.float32, copy=False)
    xq_f32 = xq.astype(np.float32, copy=False)
    d = xb_f32.shape[1]

    # Initialize the exact brute-force index based on selected metric
    if metric == faiss.METRIC_INNER_PRODUCT:
        gt = faiss.IndexFlatIP(d) # Max Inner Product (Cosine if normalized)
    elif metric == faiss.METRIC_L2:
        gt = faiss.IndexFlatL2(d) # Euclidean distance
    else:
        raise ValueError(f"Unsupported metric: {metric}")

    # Add reference database vectors and run the exact search
    gt.add(xb_f32)
    D_gt, I_gt = gt.search(xq_f32, k)
    return D_gt, I_gt


def benchmark_index(
    index: faiss.Index, xq: np.ndarray, k: int
) -> tuple[np.ndarray, np.ndarray, float, float]:
    """Search an index and return distances, ids, total time, and QPS."""
    # Input Validation
    if k <= 0:
        raise ValueError("`k` must be positive.")
    if xq.ndim != 2:
        raise ValueError("`xq` must be a 2D array.")

    xq_f32 = xq.astype(np.float32, copy=False)

    # Time the approximate index search latency
    t0 = time.perf_counter()
    D, I = index.search(xq_f32, k)
    t1 = time.perf_counter()

    # Calculate duration and throughput (Queries Per Second)
    total_time = t1 - t0
    qps = xq_f32.shape[0] / total_time if total_time > 0 else float("inf")
    return D, I, total_time, qps


# From src/indexes.py
def _build_factory_index(
    xb: np.ndarray, d: int, metric: int, factory: str
) -> faiss.Index:
    """Build, train (if needed), and populate a FAISS factory index."""
    # Input Validation
    if d <= 0:
        raise ValueError("`d` must be positive.")
    if xb.ndim != 2:
        raise ValueError("`xb` must be a 2D array.")
    if xb.shape[1] != d:
        raise ValueError(f"`d` must match xb.shape[1]. Got d={d}, xb.shape[1]={xb.shape[1]}.")
    if not factory:
        raise ValueError("`factory` cannot be empty.")

    # Ensure memory-view alignment with float32 for Faiss
    xb_f32 = xb.astype(np.float32, copy=False)
    # Initialize index using the factory string description
    index = faiss.index_factory(d, factory, metric)

    # Train index (e.g., cluster centroids or quantization codebooks) if required
    if not index.is_trained:
        index.train(xb_f32)

    # Populate the index with the base vectors
    index.add(xb_f32)
    return index


def build_hnswsq_index(
    xb: np.ndarray, d: int, metric: int, factory: str
) -> faiss.Index:
    """Build an HNSWSQ index from a FAISS factory string."""
    # Wraps the factory build helper for Hierarchical Navigable Small World + Scalar Quantization indices
    return _build_factory_index(xb=xb, d=d, metric=metric, factory=factory)


def build_lsh_index(d: int, nbits: int) -> faiss.Index:
    """Build an LSH index with explicit hash-bit count."""
    if d <= 0:
        raise ValueError("`d` must be positive.")
    if nbits <= 0:
        raise ValueError("`nbits` must be positive.")
    # Direct initialization of Locality-Sensitive Hashing index (no factory string needed)
    return faiss.IndexLSH(d, nbits)


def set_ivfpq_nprobe(index: faiss.Index, nprobe: int) -> None:
    """Set the IVFPQ search parameter `nprobe`."""
    if nprobe <= 0:
        raise ValueError("`nprobe` must be positive.")
    # ParameterSpace safely applies runtime configurations to composite/factory indices
    ps = faiss.ParameterSpace()
    # nprobe: Controls how many coarse clusters are scanned during query search
    ps.set_index_parameter(index, "nprobe", int(nprobe))


def set_hnswsq_efsearch(index: faiss.Index, ef_search: int) -> None:
    """Set the HNSWSQ search parameter `efSearch`."""
    if ef_search <= 0:
        raise ValueError("`ef_search` must be positive.")
    ps = faiss.ParameterSpace()
    # efSearch: Size of the dynamic candidate list during graph traversal at search time
    ps.set_index_parameter(index, "efSearch", int(ef_search))


# From src/rc_analysis.py
def compute_rc(xb: np.ndarray, xq: np.ndarray, k: int, metric: int) -> np.ndarray:
    """Compute Relative Contrast (RC) for each query vector.

    RC_k(q) = d_mean(q) / r_k(q)
    For L2:  r_k = true Euclidean distance to the k-th NN (sqrt of FAISS squared dist);
             d_mean estimated from a 50K random sample of xb using exact sqrt distances.
    For IP:  cosine distance (1 - inner product) used consistently for both r_k and d_mean.
    """
    # Input Validation
    if k <= 0:
        raise ValueError("`k` must be positive.")
    if xb.ndim != 2 or xq.ndim != 2:
        raise ValueError("`xb` and `xq` must be 2D arrays.")
    if xb.shape[1] != xq.shape[1]:
        raise ValueError(
            "`xb` and `xq` must have the same dimensionality. "
            f"Got {xb.shape[1]} and {xq.shape[1]}."
        )

    xb_f32 = xb.astype(np.float32, copy=False)
    xq_f32 = xq.astype(np.float32, copy=False)
    d = xb_f32.shape[1]

    # Initialize exact brute-force index
    if metric == faiss.METRIC_INNER_PRODUCT:
        gt = faiss.IndexFlatIP(d)
    elif metric == faiss.METRIC_L2:
        gt = faiss.IndexFlatL2(d)
    else:
        raise ValueError(f"Unsupported metric: {metric}")

    # Compute top-k exact neighbors to find the k-th neighbor distance
    gt.add(xb_f32)
    Dk, _ = gt.search(xq_f32, k)

    # Sample up to 50K base vectors to keep d_mean computation memory-feasible
    _RC_SAMPLE = 50_000
    rng_rc = np.random.default_rng(42)
    sample_idx = rng_rc.choice(
        xb_f32.shape[0], size=min(_RC_SAMPLE, xb_f32.shape[0]), replace=False
    )
    xb_sample = xb_f32[sample_idx]

    if metric == faiss.METRIC_L2:
        # r_k: true Euclidean distance (FAISS returns squared L2, so take sqrt)
        rk = np.sqrt(np.maximum(Dk[:, k - 1], 0.0))
        # d_mean: mean Euclidean distance from each query to sampled base vectors
        xb_sq = np.sum(xb_sample ** 2, axis=1) # (sample,)
        xq_sq = np.sum(xq_f32 ** 2, axis=1) # (n_q,)
        cross = xq_f32 @ xb_sample.T # (n_q, sample)
        sq_dists = xb_sq[None, :] + xq_sq[:, None] - 2.0 * cross
        np.maximum(sq_dists, 0.0, out=sq_dists) # clamp numerical negatives
        dmean = np.sqrt(sq_dists).mean(axis=1).astype(np.float32)
    else:
        # Cosine distance = 1 - inner product (vectors are L2-normalised)
        rk = np.maximum(1.0 - Dk[:, k - 1], 0.0)
        # d_mean = 1 - q·mean(xb_sample): exact by linearity on the unit sphere
        xb_mean_vec = np.mean(xb_sample, axis=0)
        dmean = np.maximum(1.0 - (xq_f32 @ xb_mean_vec), 0.0).astype(np.float32)

    # Avoid potential division-by-zero errors if query lies exactly on its k-th neighbor
    eps = np.finfo(np.float32).eps
    rc = dmean / np.maximum(rk, eps)
    return rc.astype(np.float32, copy=False)


def add_noise(xq: np.ndarray, sigma: float, seed: int | None = None) -> np.ndarray:
    """Add Gaussian noise to query vectors and return float32 output.

    Pass `seed` for a reproducible perturbation (recommended when saving noisy queries).
    """
    if sigma < 0:
        raise ValueError("`sigma` must be non-negative.")
    xq_f32 = xq.astype(np.float32, copy=False)
    if sigma == 0.0:
        return xq_f32.copy()

    # Generate and scale random normal distribution values to match query matrix dimensions
    rng = np.random.default_rng(seed)
    noisy = xq_f32 + sigma * rng.standard_normal(xq_f32.shape).astype(np.float32)
    return noisy.astype(np.float32, copy=False)


## Experiment Configuration

These settings define the method sweeps and the representative configurations used by the downstream analyses.


In [11]:
METHOD_ORDER = ["IVFPQ", "HNSWSQ", "LSH"]
METHOD_STYLE = {
    "IVFPQ": {"marker": "o", "color": "tab:blue"},
    "HNSWSQ": {"marker": "s", "color": "tab:orange"},
    "LSH": {"marker": "^", "color": "tab:green"},
}

BASE_IVFPQ_FACTORIES = ["IVF4096,PQ8x8", "IVF4096,PQ16x8", "IVF4096,PQ32x8"]
GLOVE_IVFPQ_FACTORIES = ["IVF4096,PQ10x8", "IVF4096,PQ20x8", "IVF4096,PQ25x8"]
HNSWSQ_FACTORIES = ["HNSW32_SQ4", "HNSW32_SQ6", "HNSW32_SQ8"]
NPROBE_VALUES = [1, 4, 8, 16, 32, 64]
EFSEARCH_VALUES = [16, 32, 64, 128, 256]
LSH_BITS = [128, 256, 512]
NOISE_SIGMAS = [0.00, 0.01, 0.05, 0.10]
SIFT_EXTENDED_NOISE_SIGMAS = [0.00, 0.10, 0.50, 1.00, 2.00]

BASE_REPRESENTATIVE_CONFIGS = [
    {"method": "IVFPQ", "factory": "IVF4096,PQ32x8", "setting_name": "nprobe", "setting_value": 64},
    {"method": "HNSWSQ", "factory": "HNSW32_SQ8", "setting_name": "efSearch", "setting_value": 256},
    {"method": "LSH", "factory": "IndexLSH", "setting_name": "nbits", "setting_value": 512},
]
GLOVE_REPRESENTATIVE_CONFIGS = [
    {"method": "IVFPQ", "factory": "IVF4096,PQ20x8", "setting_name": "nprobe", "setting_value": 64},
    {"method": "HNSWSQ", "factory": "HNSW32_SQ8", "setting_name": "efSearch", "setting_value": 256},
    {"method": "LSH", "factory": "IndexLSH", "setting_name": "nbits", "setting_value": 512},
]

GLOVE_FILE_NAME = "glove.twitter.27B.100d.txt"
GLOVE_VECTOR_DIM = 100
GLOVE_MAX_VECTORS = 200_000
GLOVE_QUERY_LIMIT = 10_000
GLOVE_MIN_QUERIES = 1_000
GLOVE_QUERY_DIVISOR = 20


## Shared Pipeline


In [12]:
def artifact_path(path: Path) -> str:
    try:
        # Convert absolute path to a clean relative path for tracking or logging
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        # Fall back to absolute string if path isn't inside the project root
        return str(path)


def save_table(df: pd.DataFrame, filename: str) -> Path:
    path = TABLES_DIR / filename
    df.to_csv(path, index=False)

    # Log path to global tracked artifacts registry if it isn't already there
    rel_path = artifact_path(path)
    if rel_path not in ARTIFACTS:
        ARTIFACTS.append(rel_path)
    return path


def save_figure(fig: plt.Figure, filename: str) -> Path:
    path = PLOTS_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig) # Free RAM memory from active matplotlib window handles

    # Log path to global tracked artifacts registry if it isn't already there
    rel_path = artifact_path(path)
    if rel_path not in ARTIFACTS:
        ARTIFACTS.append(rel_path)
    return path


def training_vectors(xb: np.ndarray, xlearn: np.ndarray | None) -> np.ndarray:
    # Use dedicated training data if provided, otherwise fall back to base vectors
    return xlearn if xlearn is not None and len(xlearn) > 0 else xb


def get_index_size_mb(index: faiss.Index) -> float:
    """Return the serialised on-disk size of a FAISS index in MiB."""
    import os
    import tempfile as _tf

    # Faiss structures don't have an accurate built-in memory sizing property,
    # so we measure memory layout by writing to a temporary file.
    fd, tmp_path = _tf.mkstemp(suffix=".faiss")
    os.close(fd)
    try:
        faiss.write_index(index, tmp_path)
        return os.path.getsize(tmp_path) / (1024.0 * 1024.0)
    finally:
        try:
            os.unlink(tmp_path) # delete file from OS storage
        except OSError:
            pass


def validate_ivfpq_factories(d: int, factories: list[str]) -> None:
    for factory in factories:
        if ",PQ" not in factory:
            continue
        # Extract subquantizer count (M) from strings like "IVF4096,PQ32x8" -> splits out 32
        m = int(factory.split(",PQ", 1)[1].split("x", 1)[0])
        # Math constraint: Dimension size must divide evenly by number of subquantizers
        if d % m != 0:
            raise ValueError(f"Invalid PQ subquantizer count in {factory} for dimension {d}.")


def build_factory_index_with_training(
    xb: np.ndarray,
    xlearn: np.ndarray | None,
    metric: int,
    factory: str,
) -> faiss.Index:
    # Initialize unconfigured base structure
    index = faiss.index_factory(xb.shape[1], factory, metric)

    # Train index codebooks (clustering/quantization maps) before vector ingestion
    if not index.is_trained:
        index.train(training_vectors(xb, xlearn).astype(np.float32, copy=False))
    index.add(xb.astype(np.float32, copy=False))
    return index


def build_configured_index(
    xb: np.ndarray,
    xlearn: np.ndarray | None,
    metric: int,
    config: dict[str, object],
) -> faiss.Index:
    method = str(config["method"])
    setting_value = int(config["setting_value"])

    # LSH handled uniquely because it doesn't support the training helper or factory definitions
    if method == "LSH":
        index = build_lsh_index(xb.shape[1], setting_value)
        index.add(xb.astype(np.float32, copy=False))
        return index

    # Build and populate standard compound index variants via factory strings
    index = build_factory_index_with_training(xb, xlearn, metric, str(config["factory"]))

    # Apply specific runtime search configurations based on algorithm architecture
    if method == "IVFPQ":
        set_ivfpq_nprobe(index, setting_value) # Number of coarse clusters to search
    elif method == "HNSWSQ":
        set_hnswsq_efsearch(index, setting_value) # Top candidate exploration depth boundary
    else:
        raise ValueError(f"Unsupported method: {method}")
    return index


def run_parameter_sweeps(
    dataset_name: str,
    xb: np.ndarray,
    xq: np.ndarray,
    xlearn: np.ndarray | None,
    I_gt: np.ndarray,
    k: int,
    metric: int,
    ivfpq_factories: list[str],
) -> pd.DataFrame:
    validate_ivfpq_factories(xb.shape[1], ivfpq_factories)
    rows: list[dict[str, object]] = []

    # 1. IVFPQ Sweep
    for factory in ivfpq_factories:
        index = build_factory_index_with_training(xb, xlearn, metric, factory)
        idx_size_mb = get_index_size_mb(index)  # measured once: nprobe doesn't change size
        for nprobe in NPROBE_VALUES:
            set_ivfpq_nprobe(index, int(nprobe))
            _, I, total_time, qps = benchmark_index(index, xq, k)
            rows.append({
                "dataset": dataset_name,
                "method": "IVFPQ",
                "factory": factory,
                "search_param_name": "nprobe",
                "search_param_value": int(nprobe),
                "recall_at_10": recall_at_k(I, I_gt, k),
                "qps": qps,
                "total_time_sec": total_time,
                "index_size_mb": idx_size_mb,
            })

    # 2. HNSWSQ Sweep
    for factory in HNSWSQ_FACTORIES:
        index = build_factory_index_with_training(xb, xlearn, metric, factory)
        idx_size_mb = get_index_size_mb(index)  # measured once: efSearch doesn't change size
        for efsearch in EFSEARCH_VALUES:
            set_hnswsq_efsearch(index, int(efsearch))
            _, I, total_time, qps = benchmark_index(index, xq, k)
            rows.append({
                "dataset": dataset_name,
                "method": "HNSWSQ",
                "factory": factory,
                "search_param_name": "efSearch",
                "search_param_value": int(efsearch),
                "recall_at_10": recall_at_k(I, I_gt, k),
                "qps": qps,
                "total_time_sec": total_time,
                "index_size_mb": idx_size_mb,
            })

    # 3. LSH Sweep
    for nbits in LSH_BITS:
        # LSH requires re-building the index for different bit counts since nbits alters structure, not search
        index = build_lsh_index(xb.shape[1], int(nbits))
        index.add(xb.astype(np.float32, copy=False))
        idx_size_mb = get_index_size_mb(index)
        _, I, total_time, qps = benchmark_index(index, xq, k)
        rows.append({
            "dataset": dataset_name,
            "method": "LSH",
            "factory": "IndexLSH",
            "search_param_name": "nbits",
            "search_param_value": int(nbits),
            "recall_at_10": recall_at_k(I, I_gt, k),
            "qps": qps,
            "total_time_sec": total_time,
            "index_size_mb": idx_size_mb,
        })

    return pd.DataFrame(rows)


def save_dataset_sweep_tables(dataset_name: str, results_df: pd.DataFrame) -> None:
    """Split the master dataframe by index type and export individual tables."""
    for method, stem in [("IVFPQ", "ivfpq"), ("HNSWSQ", "hnswsq"), ("LSH", "lsh")]:
        save_table(results_df[results_df["method"] == method].copy(), f"{dataset_name}_{stem}_results.csv")
    save_table(results_df, f"{dataset_name}_all_results.csv")


def plot_recall_qps(results_df: pd.DataFrame, dataset_name: str, filename: str | None = None) -> Path:
    """Recall@10 vs QPS trade-off curves (one curve per factory; search param varies along each curve)."""
    _ivfpq_ls = ["-", "--", ":"]
    _hnsw_ls  = ["-", "--", ":"]
    fig, ax = plt.subplots(figsize=(9, 5))

    # IVFPQ: one connected curve per PQ factory, points ordered by increasing nprobe
    ivfpq_df = results_df[results_df["method"] == "IVFPQ"]
    for i, (factory, grp) in enumerate(sorted(ivfpq_df.groupby("factory"))):
        grp = grp.sort_values("search_param_value")
        label = f"IVFPQ {factory.split(',')[1]}"
        ax.plot(grp["qps"], grp["recall_at_10"],
                marker="o", linestyle=_ivfpq_ls[i % 3],
                color=METHOD_STYLE["IVFPQ"]["color"], label=label, alpha=0.9)

    # HNSWSQ: one connected curve per SQ factory, points ordered by increasing efSearch
    hnswsq_df = results_df[results_df["method"] == "HNSWSQ"]
    for i, (factory, grp) in enumerate(sorted(hnswsq_df.groupby("factory"))):
        grp = grp.sort_values("search_param_value")
        sq_tag = factory.split("_SQ")[1] if "_SQ" in factory else factory
        label = f"HNSWSQ SQ{sq_tag}"
        ax.plot(grp["qps"], grp["recall_at_10"],
                marker="s", linestyle=_hnsw_ls[i % 3],
                color=METHOD_STYLE["HNSWSQ"]["color"], label=label, alpha=0.9)

    # LSH: discrete points (nbits is a structural param, not a runtime sweep)
    lsh_df = results_df[results_df["method"] == "LSH"].sort_values("search_param_value")
    ax.scatter(lsh_df["qps"], lsh_df["recall_at_10"],
               marker="^", color=METHOD_STYLE["LSH"]["color"],
               label="LSH (nbits ↑)", s=100, zorder=5)
    for _, row in lsh_df.iterrows():
        ax.annotate(f"{int(row['search_param_value'])}b",
                    (row["qps"], row["recall_at_10"]),
                    textcoords="offset points", xytext=(4, 4), fontsize=7)

    ax.set_xlabel("QPS")
    ax.set_ylabel("Recall@10")
    ax.set_title(f"{dataset_name.upper()}: Recall@10 vs QPS Trade-off Curves")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    return save_figure(fig, filename or f"{dataset_name}_recall_vs_qps.png")


def plot_rc_histogram(rc_df: pd.DataFrame, dataset_name: str) -> Path:
    """Plot Relative Contrast distribution to see how hard or easy the vector space is to partition."""
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(rc_df["rc"], bins=40, alpha=0.85, edgecolor="black")
    ax.set_xlabel("RC")
    ax.set_ylabel("Count")
    ax.set_title(f"{dataset_name.upper()}: RC Distribution")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    return save_figure(fig, f"{dataset_name}_rc_histogram.png")


def run_rc_analysis(dataset_name: str, xb: np.ndarray, xq: np.ndarray, k: int, metric: int) -> dict[str, object]:
    """Analyze vector space contrast and isolate easy/hard queries using interquartile ranges."""
    rc_values = compute_rc(xb, xq, k, metric)
    rc_df = pd.DataFrame({"query_id": np.arange(len(rc_values), dtype=np.int64), "rc": rc_values})

    # Calculate quantiles to split queries based on neighborhood density contrast
    q25 = float(rc_df["rc"].quantile(0.25))
    q50 = float(rc_df["rc"].quantile(0.50))
    q75 = float(rc_df["rc"].quantile(0.75))

    # Low RC implies the nearest neighbor is almost as far away as the database mean (Hard query)
    hard_query_idx = np.where(rc_values <= q25)[0]
    easy_query_idx = np.where(rc_values >= q75)[0]

    rc_stats_df = pd.DataFrame([{
        "dataset": dataset_name,
        "mean": float(rc_df["rc"].mean()),
        "std": float(rc_df["rc"].std()),
        "min": float(rc_df["rc"].min()),
        "q25": q25,
        "median": q50,
        "q75": q75,
        "max": float(rc_df["rc"].max()),
    }])
    rc_split_summary_df = pd.DataFrame([{
        "dataset": dataset_name,
        "q25": q25,
        "q75": q75,
        "hard_query_count": int(len(hard_query_idx)),
        "easy_query_count": int(len(easy_query_idx)),
        "total_query_count": int(len(rc_values)),
    }])

    save_table(rc_df, f"{dataset_name}_rc_values.csv")
    save_table(rc_split_summary_df, f"{dataset_name}_rc_split_summary.csv")
    plot_rc_histogram(rc_df, dataset_name)
    return {
        "rc_values": rc_values,
        "rc_df": rc_df,
        "rc_stats_df": rc_stats_df,
        "rc_split_summary_df": rc_split_summary_df,
        "hard_query_idx": hard_query_idx,
        "easy_query_idx": easy_query_idx,
    }


def run_easy_hard_evaluation(
    dataset_name: str,
    xb: np.ndarray,
    xq: np.ndarray,
    xlearn: np.ndarray | None,
    I_gt: np.ndarray,
    k: int,
    metric: int,
    hard_query_idx: np.ndarray,
    easy_query_idx: np.ndarray,
    representative_configs: list[dict[str, object]],
) -> pd.DataFrame:
    """Evaluate Recall@10 and QPS separately for all, hard, and easy query groups."""
    rows: list[dict[str, object]] = []
    for config in representative_configs:
        index = build_configured_index(xb, xlearn, metric, config)
        # Run benchmark separately on each subset so QPS reflects the actual per-group throughput
        _, I_all,  _, qps_all  = benchmark_index(index, xq, k)
        _, I_hard, _, qps_hard = benchmark_index(index, xq[hard_query_idx], k)
        _, I_easy, _, qps_easy = benchmark_index(index, xq[easy_query_idx], k)

        rows.append({
            "dataset": dataset_name,
            "method": str(config["method"]),
            "factory": str(config["factory"]),
            "setting_name": str(config["setting_name"]),
            "setting_value": int(config["setting_value"]),
            "recall_all":  recall_at_k(I_all,  I_gt, k),
            "recall_hard": recall_at_k(I_hard, I_gt[hard_query_idx], k),
            "recall_easy": recall_at_k(I_easy, I_gt[easy_query_idx], k),
            "qps_all":  qps_all,
            "qps_hard": qps_hard,
            "qps_easy": qps_easy,
        })
    easy_hard_df = pd.DataFrame(rows)
    save_table(easy_hard_df, f"{dataset_name}_easy_hard_results.csv")
    return easy_hard_df


def plot_easy_hard(easy_hard_df: pd.DataFrame, dataset_name: str) -> Path:
    """Grouped bar charts comparing Recall@10 and QPS across all, hard, and easy query groups."""
    methods = easy_hard_df["method"].tolist()
    x = np.arange(len(methods))
    width = 0.25
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Recall subplot
    axes[0].bar(x - width, easy_hard_df["recall_all"],  width=width, label="all")
    axes[0].bar(x,          easy_hard_df["recall_hard"], width=width, label="hard")
    axes[0].bar(x + width,  easy_hard_df["recall_easy"], width=width, label="easy")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(methods)
    axes[0].set_ylabel("Recall@10")
    axes[0].set_title(f"{dataset_name.upper()}: Recall@10")
    axes[0].grid(axis="y", alpha=0.3)
    axes[0].legend()

    # QPS subplot
    axes[1].bar(x - width, easy_hard_df["qps_all"],  width=width, label="all")
    axes[1].bar(x,          easy_hard_df["qps_hard"], width=width, label="hard")
    axes[1].bar(x + width,  easy_hard_df["qps_easy"], width=width, label="easy")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(methods)
    axes[1].set_ylabel("QPS")
    axes[1].set_title(f"{dataset_name.upper()}: QPS")
    axes[1].grid(axis="y", alpha=0.3)
    axes[1].legend()

    fig.suptitle(f"{dataset_name.upper()}: All vs Hard vs Easy Queries")
    fig.tight_layout()
    return save_figure(fig, f"{dataset_name}_easy_vs_hard_recall.png")


def run_noise_robustness(
    dataset_name: str,
    xb: np.ndarray,
    xq: np.ndarray,
    xlearn: np.ndarray | None,
    I_gt: np.ndarray,
    k: int,
    metric: int,
    representative_configs: list[dict[str, object]],
    sigma_values: list[float],
    table_filename: str,
) -> pd.DataFrame:
    """Benchmark how index accuracy decays as query vectors get progressively degraded with random noise."""
    # Pre-generate noisy queries once per sigma so all methods see identical perturbations.
    noisy_cache: dict[float, np.ndarray] = {}
    for sigma in sigma_values:
        xq_n = add_noise(xq, float(sigma), seed=RANDOM_SEED)
        if metric == faiss.METRIC_INNER_PRODUCT:
            xq_n = l2_normalize(xq_n) # Re-normalize vector magnitude to preserve Cosine mapping
        noisy_cache[sigma] = xq_n

        # Save perturbed variations to disk for compliance and reproducibility audits
        if sigma > 0.0:
            sigma_tag = f"{sigma:.2f}".replace(".", "")
            noisy_path = NOISY_QUERIES_DIR / f"{dataset_name}_noisy_sigma{sigma_tag}.npy"
            np.save(noisy_path, xq_n)

    rows: list[dict[str, object]] = []
    for config in representative_configs:
        index = build_configured_index(xb, xlearn, metric, config)
        for sigma in sigma_values:
            xq_noisy = noisy_cache[sigma]
            _, I, _, qps = benchmark_index(index, xq_noisy, k)
            rows.append({
                "dataset": dataset_name,
                "method": str(config["method"]),
                "factory": str(config["factory"]),
                "noise_sigma": float(sigma),
                "recall_at_10": recall_at_k(I, I_gt, k),
                "qps": qps,
            })
    noise_df = pd.DataFrame(rows)
    save_table(noise_df, table_filename)
    return noise_df


def plot_noise_robustness(noise_df: pd.DataFrame, dataset_name: str, filename: str, title: str | None = None) -> Path:
    """Plot line curves tracking precision degradation under increasing standard deviation of noise."""
    fig, ax = plt.subplots(figsize=(8, 5))
    for method in METHOD_ORDER:
        subset = noise_df[noise_df["method"] == method].sort_values("noise_sigma")
        ax.plot(subset["noise_sigma"], subset["recall_at_10"], marker="o", label=method)
    ax.set_xlabel("Noise sigma")
    ax.set_ylabel("Recall@10")
    ax.set_title(title or f"{dataset_name.upper()}: Recall@10 vs Query Noise")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    return save_figure(fig, filename)


def run_dataset_pipeline(
    dataset_name: str,
    xb: np.ndarray,
    xq: np.ndarray,
    xlearn: np.ndarray | None,
    metric: int,
    k: int,
    ivfpq_factories: list[str],
    representative_configs: list[dict[str, object]],
) -> dict[str, object]:
    """Run the complete evaluation workflow for a dataset from scratch."""
    print(f"{dataset_name.upper()}: building exact ground truth")
    _, I_gt = build_ground_truth(xb, xq, k, metric)

    print(f"{dataset_name.upper()}: running method sweeps")
    results_df = run_parameter_sweeps(dataset_name, xb, xq, xlearn, I_gt, k, metric, ivfpq_factories)
    save_dataset_sweep_tables(dataset_name, results_df)
    plot_recall_qps(results_df, dataset_name)

    print(f"{dataset_name.upper()}: running RC/easy-hard/noise analyses")
    rc_outputs = run_rc_analysis(dataset_name, xb, xq, k, metric)

    # Run topology partitioning evaluations
    easy_hard_df = run_easy_hard_evaluation(
        dataset_name,
        xb,
        xq,
        xlearn,
        I_gt,
        k,
        metric,
        rc_outputs["hard_query_idx"],
        rc_outputs["easy_query_idx"],
        representative_configs,
    )
    plot_easy_hard(easy_hard_df, dataset_name)

    # Run mutation evaluations
    noise_df = run_noise_robustness(
        dataset_name,
        xb,
        xq,
        xlearn,
        I_gt,
        k,
        metric,
        representative_configs,
        NOISE_SIGMAS,
        f"{dataset_name}_noisy_query_results.csv",
    )
    plot_noise_robustness(noise_df, dataset_name, f"{dataset_name}_noise_vs_recall.png")

    print(f"{dataset_name.upper()}: complete ({len(results_df)} sweep rows)")
    return {
        "dataset": dataset_name,
        "xb": xb,
        "xq": xq,
        "xlearn": xlearn,
        "metric": metric,
        "k": k,
        "I_gt": I_gt,
        "results_df": results_df,
        "easy_hard_df": easy_hard_df,
        "noise_df": noise_df,
        **rc_outputs,
    }


## GloVe Loader


In [6]:
def load_glove_vectors(
    max_vectors: int | None = GLOVE_MAX_VECTORS,
    vector_dim: int = GLOVE_VECTOR_DIM,
    seed: int = RANDOM_SEED,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, None]:
    glove_txt_path = RAW_DATA_ROOT / "glove" / GLOVE_FILE_NAME
    rows: list[np.ndarray] = []

    # Parse Raw GloVe Text File
    with glove_txt_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            # Locate the space separating the word string token from the numerical vector values
            first_space = line.find(" ")
            if first_space == -1:
                continue # Skip malformed lines

            # Skip the word text entirely and parse only the remaining numbers into a float array
            vec = np.fromstring(line[first_space + 1 :], sep=" ", dtype=np.float32)
            if vec.size != vector_dim:
                continue
            rows.append(vec)
            if max_vectors is not None and len(rows) >= max_vectors:
                break

    if not rows:
        raise ValueError(f"No valid GloVe vectors parsed from: {glove_txt_path}")

    # Stack the parsed rows into a uniform 2D matrix
    vectors = np.vstack(rows).astype(np.float32, copy=False)

    # Calculate query count dynamically based on the volume of parsed vectors
    n_query = min(GLOVE_QUERY_LIMIT, max(GLOVE_MIN_QUERIES, len(vectors) // GLOVE_QUERY_DIVISOR))
    if n_query >= len(vectors):
        raise ValueError("Not enough vectors to create a non-overlapping GloVe base/query split.")

    # Random Dataset Partitioning
    rng = np.random.default_rng(seed)
    permutation = rng.permutation(len(vectors))

    # Slice the randomly shuffled indices to isolate mutually exclusive query and base pools
    query_idx = permutation[:n_query]
    base_idx = permutation[n_query:]

    # Apply L2-normalization to adapt vectors for Cosine searches via Inner Product indices
    xb = vectors[base_idx].astype(np.float32, copy=False)
    xq = vectors[query_idx].astype(np.float32, copy=False)
    xb = l2_normalize(xb)
    xq = l2_normalize(xq)

    # Returns (base, queries, training_set, ground_truth)
    # Note: xb is used as the fallback training set: ground_truth is None and computed later
    return xb, xq, xb, None


## SIFT Experiments


In [7]:
# Execution Driver for the SIFT Datase

# Load the structured SIFT1M vector groups (Base, Query, and Quantizer Learn splits)
xb_sift, xq_sift, xlearn_sift, _ = load_sift()

# Run the complete experimental evaluation pipeline for SIFT
sift_output = run_dataset_pipeline(
    dataset_name="sift",
    xb=xb_sift,
    xq=xq_sift,
    xlearn=xlearn_sift,
    metric=faiss.METRIC_L2, # SIFT features use Euclidean distance
    k=10, # Evaluate performance for top-10 nearest neighbors
    ivfpq_factories=BASE_IVFPQ_FACTORIES,
    representative_configs=BASE_REPRESENTATIVE_CONFIGS,
)

# Extract structured evaluation results for downstream reporting and analysis
sift_results_df = sift_output["results_df"] # Complete grid sweep data (Recall, QPS, Size)
sift_easy_hard_df = sift_output["easy_hard_df"] # Accuracy filtered by vector neighborhood contrast
sift_noise_df = sift_output["noise_df"] # Accuracy degradation under Gaussian mutations


SIFT: building exact ground truth
SIFT: running method sweeps
SIFT: running RC/easy-hard/noise analyses
SIFT: complete (36 sweep rows)


## GIST Experiments


In [8]:
xb_gist, xq_gist, xlearn_gist, _ = load_gist()
gist_output = run_dataset_pipeline(
    dataset_name="gist",
    xb=xb_gist,
    xq=xq_gist,
    xlearn=xlearn_gist,
    metric=faiss.METRIC_L2,
    k=10,
    ivfpq_factories=BASE_IVFPQ_FACTORIES,
    representative_configs=BASE_REPRESENTATIVE_CONFIGS,
)
gist_results_df = gist_output["results_df"]
gist_easy_hard_df = gist_output["easy_hard_df"]
gist_noise_df = gist_output["noise_df"]


GIST: building exact ground truth
GIST: running method sweeps
GIST: running RC/easy-hard/noise analyses
GIST: complete (36 sweep rows)


## GloVe Experiments


In [9]:
xb_glove, xq_glove, xlearn_glove, _ = load_glove_vectors()
glove_output = run_dataset_pipeline(
    dataset_name="glove",
    xb=xb_glove,
    xq=xq_glove,
    xlearn=xlearn_glove,
    metric=faiss.METRIC_INNER_PRODUCT,
    k=10,
    ivfpq_factories=GLOVE_IVFPQ_FACTORIES,
    representative_configs=GLOVE_REPRESENTATIVE_CONFIGS,
)
glove_results_df = glove_output["results_df"]
glove_easy_hard_df = glove_output["easy_hard_df"]
glove_noise_df = glove_output["noise_df"]


GLOVE: building exact ground truth
GLOVE: running method sweeps
GLOVE: running RC/easy-hard/noise analyses
GLOVE: complete (36 sweep rows)


## Cross-Dataset Summary


In [10]:
def plot_best_recall_by_method(all_results_df: pd.DataFrame) -> Path:
    best_recall_method_dataset_df = all_results_df.groupby(["dataset", "method"], as_index=False)["recall_at_10"].max()
    dataset_order = ["sift", "gist", "glove"]
    pivot_best = best_recall_method_dataset_df.pivot(index="dataset", columns="method", values="recall_at_10").reindex(dataset_order)
    datasets = pivot_best.index.tolist()
    methods = [method for method in METHOD_ORDER if method in pivot_best.columns]
    x = np.arange(len(datasets))
    width = 0.25
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, method in enumerate(methods):
        offset = (i - (len(methods) - 1) / 2) * width
        ax.bar(x + offset, pivot_best[method].values, width=width, label=method)
    ax.set_xticks(x)
    ax.set_xticklabels(datasets)
    ax.set_ylabel("Best Recall@10")
    ax.set_title("Best Recall@10 by Method and Dataset")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    return save_figure(fig, "best_recall_by_method_dataset.png")


def plot_noise_comparison(all_noisy_query_results_df: pd.DataFrame) -> Path:
    dataset_order = ["sift", "gist", "glove"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, dataset_name in zip(axes, dataset_order):
        ds_df = all_noisy_query_results_df[all_noisy_query_results_df["dataset"] == dataset_name]
        for method in METHOD_ORDER:
            subset = ds_df[ds_df["method"] == method].sort_values("noise_sigma")
            ax.plot(subset["noise_sigma"], subset["recall_at_10"], marker="o", label=method)
        ax.set_title(dataset_name.upper())
        ax.set_xlabel("Noise sigma")
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("Recall@10")
    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3)
    fig.suptitle("Noise Robustness Comparison Across Datasets", y=1.03)
    fig.tight_layout()
    return save_figure(fig, "noise_robustness_comparison.png")


def plot_easy_hard_comparison(all_easy_hard_results_df: pd.DataFrame) -> Path:
    """Cross-dataset grouped bar charts for Recall@10 and QPS across all, hard, and easy groups."""
    plot_df = all_easy_hard_results_df.copy()
    plot_df["label"] = plot_df["dataset"].str.upper() + "-" + plot_df["method"]
    x = np.arange(len(plot_df))
    width = 0.25
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # Recall subplot
    axes[0].bar(x - width, plot_df["recall_all"],  width=width, label="all")
    axes[0].bar(x,          plot_df["recall_hard"], width=width, label="hard")
    axes[0].bar(x + width,  plot_df["recall_easy"], width=width, label="easy")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(plot_df["label"], rotation=30, ha="right")
    axes[0].set_ylabel("Recall@10")
    axes[0].set_title("Recall@10: All vs Hard vs Easy Queries")
    axes[0].grid(axis="y", alpha=0.3)
    axes[0].legend()

    # QPS subplot
    axes[1].bar(x - width, plot_df["qps_all"],  width=width, label="all")
    axes[1].bar(x,          plot_df["qps_hard"], width=width, label="hard")
    axes[1].bar(x + width,  plot_df["qps_easy"], width=width, label="easy")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(plot_df["label"], rotation=30, ha="right")
    axes[1].set_ylabel("QPS")
    axes[1].set_title("QPS: All vs Hard vs Easy Queries")
    axes[1].grid(axis="y", alpha=0.3)
    axes[1].legend()

    fig.suptitle("Easy/Hard Recall and QPS Comparison Across Datasets and Methods")
    fig.tight_layout()
    return save_figure(fig, "easy_hard_comparison.png")


def build_cross_dataset_summary(outputs: dict[str, dict[str, object]]) -> dict[str, pd.DataFrame]:
    all_results_df = pd.concat([outputs[name]["results_df"] for name in outputs], ignore_index=True)
    all_results_sorted_df = all_results_df.sort_values(["dataset", "recall_at_10", "qps"], ascending=[True, False, False]).reset_index(drop=True)
    save_table(all_results_df, "all_datasets_results.csv")

    best_config_per_dataset_method_df = (
        all_results_df
        .sort_values(["dataset", "method", "recall_at_10", "qps"], ascending=[True, True, False, False])
        .groupby(["dataset", "method"], as_index=False)
        .first()
    )
    save_table(best_config_per_dataset_method_df, "best_config_per_dataset_method.csv")

    best_overall_per_dataset_df = (
        all_results_df
        .sort_values(["dataset", "recall_at_10", "qps"], ascending=[True, False, False])
        .groupby("dataset", as_index=False)
        .first()
    )
    save_table(best_overall_per_dataset_df, "best_overall_per_dataset.csv")

    high_qps_summary_df = (
        all_results_df
        .sort_values(["dataset", "method", "qps", "recall_at_10"], ascending=[True, True, False, False])
        .groupby(["dataset", "method"], as_index=False)
        .first()
    )
    save_table(high_qps_summary_df, "high_qps_summary.csv")

    for name, context in outputs.items():
        plot_recall_qps(context["results_df"], name, f"{name}_recall_vs_qps_final.png")
    plot_best_recall_by_method(all_results_df)

    all_noisy_query_results_df = pd.concat([outputs[name]["noise_df"] for name in outputs], ignore_index=True)
    save_table(all_noisy_query_results_df, "all_noisy_query_results.csv")
    plot_noise_comparison(all_noisy_query_results_df)

    all_easy_hard_results_df = pd.concat([outputs[name]["easy_hard_df"] for name in outputs], ignore_index=True)
    save_table(all_easy_hard_results_df, "all_easy_hard_results.csv")
    plot_easy_hard_comparison(all_easy_hard_results_df)

    return {
        "all_results_df": all_results_df,
        "all_results_sorted_df": all_results_sorted_df,
        "best_config_per_dataset_method_df": best_config_per_dataset_method_df,
        "best_overall_per_dataset_df": best_overall_per_dataset_df,
        "high_qps_summary_df": high_qps_summary_df,
        "all_noisy_query_results_df": all_noisy_query_results_df,
        "all_easy_hard_results_df": all_easy_hard_results_df,
    }


dataset_outputs = {"sift": sift_output, "gist": gist_output, "glove": glove_output}
cross_dataset_outputs = build_cross_dataset_summary(dataset_outputs)

all_results_df = cross_dataset_outputs["all_results_df"]
best_config_per_dataset_method_df = cross_dataset_outputs["best_config_per_dataset_method_df"]
best_overall_per_dataset_df = cross_dataset_outputs["best_overall_per_dataset_df"]
high_qps_summary_df = cross_dataset_outputs["high_qps_summary_df"]
all_noisy_query_results_df = cross_dataset_outputs["all_noisy_query_results_df"]
all_easy_hard_results_df = cross_dataset_outputs["all_easy_hard_results_df"]

best_config_per_dataset_method_df


,dataset,method,factory,search_param_name,search_param_value,recall_at_10,qps,total_time_sec,index_size_mb
0,gist,HNSWSQ,HNSW32_SQ8,efSearch,256,0.94190,2088.740996,0.478757,1175.057623
1,gist,IVFPQ,"IVF4096,PQ32x8",nprobe,64,0.27560,1483.732799,0.673976,54.115894
2,gist,LSH,IndexLSH,nbits,512,0.07220,1268.588629,0.788278,62.910245
3,glove,HNSWSQ,HNSW32_SQ8,efSearch,256,0.97748,8971.226406,1.114675,67.428747
4,glove,IVFPQ,"IVF4096,PQ25x8",nprobe,64,0.71862,49284.803561,0.202902,7.671116
5,glove,LSH,IndexLSH,nbits,512,0.60093,12406.653887,0.806019,11.792081
6,sift,HNSWSQ,HNSW32_SQ8,efSearch,256,0.98690,2180.745747,4.585587,381.594244
7,sift,IVFPQ,"IVF4096,PQ32x8",nprobe,64,0.71881,8379.815471,1.193344,40.303394
8,sift,LSH,IndexLSH,nbits,512,0.38540,760.963903,13.141228,61.285245


## Adaptive Search Effort on Hard Queries


In [11]:
def plot_adaptive_hard_query(adaptive_hard_query_df: pd.DataFrame) -> Path:
    plot_df = adaptive_hard_query_df.copy()
    plot_df["dataset_method"] = plot_df["dataset"].str.upper() + "-" + plot_df["method"]
    min_val = plot_df.groupby(["dataset", "method"])["search_param_value"].transform("min")
    plot_df["effort_level"] = np.where(plot_df["search_param_value"] == min_val, "low", "high")
    order = (
        plot_df[["dataset", "method", "dataset_method"]]
        .drop_duplicates()
        .sort_values(["dataset", "method"])
        ["dataset_method"]
        .tolist()
    )
    pivot_recall = plot_df.pivot(index="dataset_method", columns="effort_level", values="recall_hard").reindex(order)
    pivot_qps = plot_df.pivot(index="dataset_method", columns="effort_level", values="qps_hard").reindex(order)

    x = np.arange(len(order))
    width = 0.35
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(x - width / 2, pivot_recall["low"], width=width, label="low effort")
    axes[0].bar(x + width / 2, pivot_recall["high"], width=width, label="high effort")
    axes[0].set_title("Hard-query Recall@10")
    axes[0].set_ylabel("Recall@10")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(order, rotation=30, ha="right")
    axes[0].grid(axis="y", alpha=0.3)
    axes[0].legend()

    axes[1].bar(x - width / 2, pivot_qps["low"], width=width, label="low effort")
    axes[1].bar(x + width / 2, pivot_qps["high"], width=width, label="high effort")
    axes[1].set_title("Hard-query QPS")
    axes[1].set_ylabel("QPS")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(order, rotation=30, ha="right")
    axes[1].grid(axis="y", alpha=0.3)

    fig.suptitle("Adaptive Search Effort on Hard Queries", y=1.03)
    fig.tight_layout()
    return save_figure(fig, "adaptive_hard_query_comparison.png")


def run_adaptive_hard_query_analysis(outputs: dict[str, dict[str, object]]) -> pd.DataFrame:
    adaptive_specs = [
        {"dataset": "sift", "method": "IVFPQ", "factory": "IVF4096,PQ32x8", "search_param_name": "nprobe", "low": 4, "high": 64},
        {"dataset": "sift", "method": "HNSWSQ", "factory": "HNSW32_SQ8", "search_param_name": "efSearch", "low": 32, "high": 256},
        {"dataset": "gist", "method": "IVFPQ", "factory": "IVF4096,PQ32x8", "search_param_name": "nprobe", "low": 4, "high": 64},
        {"dataset": "gist", "method": "HNSWSQ", "factory": "HNSW32_SQ8", "search_param_name": "efSearch", "low": 32, "high": 256},
        {"dataset": "glove", "method": "IVFPQ", "factory": "IVF4096,PQ25x8", "search_param_name": "nprobe", "low": 4, "high": 64},
        {"dataset": "glove", "method": "HNSWSQ", "factory": "HNSW32_SQ8", "search_param_name": "efSearch", "low": 32, "high": 256},
    ]
    rows: list[dict[str, object]] = []
    for spec in adaptive_specs:
        context = outputs[str(spec["dataset"])]
        xb = context["xb"]
        xq = context["xq"]
        xlearn = context["xlearn"]
        metric = int(context["metric"])
        k = int(context["k"])
        I_gt = context["I_gt"]
        hard_idx = context["hard_query_idx"]
        index = build_factory_index_with_training(xb, xlearn, metric, str(spec["factory"]))

        for effort_value in [int(spec["low"]), int(spec["high"])]:
            if spec["search_param_name"] == "nprobe":
                set_ivfpq_nprobe(index, effort_value)
            else:
                set_hnswsq_efsearch(index, effort_value)
            _, I, _, qps = benchmark_index(index, xq[hard_idx], k)
            rows.append({
                "dataset": spec["dataset"],
                "method": spec["method"],
                "factory": spec["factory"],
                "search_param_name": spec["search_param_name"],
                "search_param_value": effort_value,
                "recall_hard": recall_at_k(I, I_gt[hard_idx], k),
                "qps_hard": qps,
            })
    adaptive_hard_query_df = pd.DataFrame(rows)
    save_table(adaptive_hard_query_df, "adaptive_hard_query_results.csv")
    plot_adaptive_hard_query(adaptive_hard_query_df)
    return adaptive_hard_query_df


adaptive_hard_query_df = run_adaptive_hard_query_analysis(dataset_outputs)
adaptive_hard_query_df


,dataset,method,factory,search_param_name,search_param_value,recall_hard,qps_hard
0,sift,IVFPQ,"IVF4096,PQ32x8",nprobe,4,0.40608,65772.511271
1,sift,IVFPQ,"IVF4096,PQ32x8",nprobe,64,0.66184,9429.777579
2,sift,HNSWSQ,HNSW32_SQ8,efSearch,32,0.87520,18856.425669
3,sift,HNSWSQ,HNSW32_SQ8,efSearch,256,0.98384,4968.156107
4,gist,IVFPQ,"IVF4096,PQ32x8",nprobe,4,0.13600,9901.656771
5,gist,IVFPQ,"IVF4096,PQ32x8",nprobe,64,0.21400,4761.596390
6,gist,HNSWSQ,HNSW32_SQ8,efSearch,32,0.56360,4278.418970
7,gist,HNSWSQ,HNSW32_SQ8,efSearch,256,0.90080,937.765388
8,glove,IVFPQ,"IVF4096,PQ25x8",nprobe,4,0.38968,91037.536527
9,glove,IVFPQ,"IVF4096,PQ25x8",nprobe,64,0.61916,32518.675500


## Extended SIFT Noise Sensitivity


In [12]:
sift_extended_noise_df = run_noise_robustness(
    dataset_name="sift",
    xb=sift_output["xb"],
    xq=sift_output["xq"],
    xlearn=sift_output["xlearn"],
    I_gt=sift_output["I_gt"],
    k=sift_output["k"],
    metric=sift_output["metric"],
    representative_configs=BASE_REPRESENTATIVE_CONFIGS,
    sigma_values=SIFT_EXTENDED_NOISE_SIGMAS,
    table_filename="sift_extended_noise_results.csv",
)
plot_noise_robustness(
    sift_extended_noise_df,
    "sift",
    "sift_extended_noise_vs_recall.png",
    title="SIFT Extended Noise Sensitivity",
)
sift_extended_noise_df


,dataset,method,factory,noise_sigma,recall_at_10,qps
0,sift,IVFPQ,"IVF4096,PQ32x8",0.0,0.71881,13104.576353
1,sift,IVFPQ,"IVF4096,PQ32x8",0.1,0.71868,8714.817953
2,sift,IVFPQ,"IVF4096,PQ32x8",0.5,0.71779,9272.617839
3,sift,IVFPQ,"IVF4096,PQ32x8",1.0,0.71694,7890.630184
4,sift,IVFPQ,"IVF4096,PQ32x8",2.0,0.71096,9632.914667
5,sift,HNSWSQ,HNSW32_SQ8,0.0,0.98674,7676.815289
6,sift,HNSWSQ,HNSW32_SQ8,0.1,0.98619,10365.967367
7,sift,HNSWSQ,HNSW32_SQ8,0.5,0.97519,10365.059467
8,sift,HNSWSQ,HNSW32_SQ8,1.0,0.95830,10465.058843
9,sift,HNSWSQ,HNSW32_SQ8,2.0,0.92353,9823.513700


## Final Tables and Artifacts


In [13]:
rc_stats_df = pd.concat([
    sift_output["rc_stats_df"],
    gist_output["rc_stats_df"],
    glove_output["rc_stats_df"],
], ignore_index=True)

final_summary = {
    "best_config_per_dataset_method": best_config_per_dataset_method_df,
    "best_overall_per_dataset": best_overall_per_dataset_df,
    "high_qps_summary": high_qps_summary_df,
    "rc_stats": rc_stats_df,
    "artifacts": pd.DataFrame({"path": ARTIFACTS}),
}

final_summary["best_overall_per_dataset"]


,dataset,method,factory,search_param_name,search_param_value,recall_at_10,qps,total_time_sec,index_size_mb
0,gist,HNSWSQ,HNSW32_SQ8,efSearch,256,0.94190,2088.740996,0.478757,1175.057623
1,glove,HNSWSQ,HNSW32_SQ8,efSearch,256,0.97748,8971.226406,1.114675,67.428747
2,sift,HNSWSQ,HNSW32_SQ8,efSearch,256,0.98690,2180.745747,4.585587,381.594244


In [14]:
# ─── Save Verification & Archive ─────────────────────────────────────────────
import os

print("=" * 60)
print("SAVED ARTIFACTS")
print("=" * 60)
total_size_mb = 0.0
missing = []
for art in sorted(ARTIFACTS):
    full_path = PROJECT_ROOT / art
    if full_path.exists():
        size_mb = full_path.stat().st_size / (1024 * 1024)
        total_size_mb += size_mb
        print(f"  [OK] {art:60s} {size_mb:7.2f} MB")
    else:
        missing.append(art)
        print(f"  [!!] MISSING: {art}")

print()
print("SAVED NOISY QUERY VECTORS")
print("=" * 60)
for p in sorted(NOISY_QUERIES_DIR.glob("*.npy")):
    size_mb = p.stat().st_size / (1024 * 1024)
    total_size_mb += size_mb
    rel = str(p.relative_to(PROJECT_ROOT))
    print(f"  [OK] {rel:60s} {size_mb:7.2f} MB")

print()
print(f"Total persisted : {total_size_mb:.2f} MB  |  Missing files: {len(missing)}")
print(f"All results saved locally at: {PROJECT_ROOT}")


SAVED ARTIFACTS
  [OK] results\plots\adaptive_hard_query_comparison.png                0.07 MB
  [OK] results\plots\best_recall_by_method_dataset.png                 0.03 MB
  [OK] results\plots\easy_hard_comparison.png                          0.13 MB
  [OK] results\plots\gist_easy_vs_hard_recall.png                      0.05 MB
  [OK] results\plots\gist_noise_vs_recall.png                          0.06 MB
  [OK] results\plots\gist_rc_histogram.png                             0.03 MB
  [OK] results\plots\gist_recall_vs_qps.png                            0.08 MB
  [OK] results\plots\gist_recall_vs_qps_final.png                      0.08 MB
  [OK] results\plots\glove_easy_vs_hard_recall.png                     0.04 MB
  [OK] results\plots\glove_noise_vs_recall.png                         0.07 MB
  [OK] results\plots\glove_rc_histogram.png                            0.03 MB
  [OK] results\plots\glove_recall_vs_qps.png                           0.08 MB
  [OK] results\plots\glove_recall_vs

## Poster Figures

Merge the three result DataFrames into tidy CSV exports, then produce the five
poster figures saved to `results/figures/` at 300 dpi.
Run these cells any time after the experiment cells have completed (or after
`results/tables/` CSVs exist from a previous run).

In [13]:
# --- Poster Figure Setup ------------------------------------------------------
import matplotlib as mpl
mpl.rcParams.update({"font.family": "serif"})

FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

_POSTER_COLORS = {
    "IVFPQ":  "#2D5A27",   # dark forest green
    "HNSWSQ": "#8B7D3A",   # muted olive
    "LSH":    "#8F9779",   # sage
}
_FACTORY_SHADES = {
    "IVFPQ":  ["#1a3610", "#2D5A27", "#4a8a3d"],
    "HNSWSQ": ["#5c5225", "#8B7D3A", "#b5a34c"],
    "LSH":    ["#8F9779"],
}
_DATASET_LABELS = {"sift": "SIFT1M", "gist": "GIST1M", "glove": "GloVe-100d"}
_DATASETS       = ["sift", "gist", "glove"]


def _load_df(varname: str, csv_rel: str) -> pd.DataFrame:
    # Return in-memory DataFrame if present, else load from the saved CSV.
    g = globals()
    if varname in g and isinstance(g[varname], pd.DataFrame):
        return g[varname].copy()
    path = PROJECT_ROOT / "results" / csv_rel
    if path.exists():
        return pd.read_csv(path)
    raise FileNotFoundError(
        f"'{varname}' not in memory and '{path}' not found -- "
        "re-run the experiment cells first."
    )


def _save_poster_fig(fig, filename: str) -> None:
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved -> {path.relative_to(PROJECT_ROOT)}")


# Load source DataFrames (memory-first, CSV fallback)
sweep_df = _load_df("all_results_df",            "tables/all_datasets_results.csv")
eh_df    = _load_df("all_easy_hard_results_df",  "tables/all_easy_hard_results.csv")
noise_df = _load_df("all_noisy_query_results_df","tables/all_noisy_query_results.csv")

sweep_df = sweep_df.rename(columns={"index_size_mb": "memory_mb"})

# Persist tidy unified exports
(PROJECT_ROOT / "results").mkdir(parents=True, exist_ok=True)
sweep_df.to_csv(PROJECT_ROOT / "results" / "results.csv",   index=False)
eh_df.to_csv(   PROJECT_ROOT / "results" / "easy_hard.csv", index=False)
noise_df.to_csv(PROJECT_ROOT / "results" / "noise.csv",     index=False)
print(f"results.csv   : {len(sweep_df)} rows  (index_size_mb renamed to memory_mb)")
print(f"easy_hard.csv : {len(eh_df)} rows")
print(f"noise.csv     : {len(noise_df)} rows")

results.csv   : 108 rows  (index_size_mb renamed to memory_mb)
easy_hard.csv : 9 rows
noise.csv     : 36 rows


In [ ]:
# --- Figure 1 : Recall-QPS Pareto Curves -------------------------------------

def _pareto_mask(recalls: np.ndarray, qps_vals: np.ndarray) -> np.ndarray:
    # True where no other point dominates (>= both axes, strict in at least one).
    mask = np.ones(len(recalls), dtype=bool)
    for i in range(len(recalls)):
        for j in range(len(recalls)):
            if i != j and recalls[j] >= recalls[i] and qps_vals[j] >= qps_vals[i]:
                if recalls[j] > recalls[i] or qps_vals[j] > qps_vals[i]:
                    mask[i] = False
                    break
    return mask


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor("#f3fadc")
for _ax in axes:
    _ax.set_facecolor("#f3fadc")

for ax, dataset in zip(axes, _DATASETS):
    ds_df = sweep_df[sweep_df["dataset"] == dataset]
    for method in METHOD_ORDER:
        m_df = ds_df[ds_df["method"] == method].sort_values("recall_at_10")
        if m_df.empty:
            continue
        r = m_df["recall_at_10"].values
        q = m_df["qps"].values
        c = _POSTER_COLORS[method]

        # All sweep points (faint background)
        ax.scatter(r, q, color=c, alpha=0.30, s=22, zorder=2)
        ax.plot(r, q, color=c, alpha=0.20, linewidth=0.8, zorder=1)

        # Pareto-optimal subset (highlighted)
        pm    = _pareto_mask(r, q)
        ax.scatter(r[pm], q[pm], color=c, s=65, zorder=4,
                   label=method, edgecolors="white", linewidth=0.6)
        order = np.argsort(r[pm])
        ax.plot(r[pm][order], q[pm][order], color=c, linewidth=2.0, zorder=3)

    ax.set_yscale("log")
    ax.set_xlabel("Recall@10", fontsize=11)
    ax.set_ylabel("QPS (log)", fontsize=11)
    ax.set_title(_DATASET_LABELS[dataset], fontsize=13)
    ax.grid(alpha=0.25, which="both")
    ax.legend(fontsize=9)

fig.suptitle("Recall-QPS Pareto Curves", fontsize=15, y=1.01)
fig.tight_layout()
_save_poster_fig(fig, "pareto_recall_qps.png")

Saved -> results\figures\pareto_recall_qps.png


In [15]:
# --- Figure 2 : Index Memory vs Recall@10 ------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, dataset in zip(axes, _DATASETS):
    ds_df = sweep_df[sweep_df["dataset"] == dataset]
    for method in METHOD_ORDER:
        m_df = ds_df[ds_df["method"] == method]
        if m_df.empty:
            continue
        ax.scatter(m_df["recall_at_10"], m_df["memory_mb"],
                   color=_POSTER_COLORS[method], label=method,
                   alpha=0.75, s=40, edgecolors="white", linewidth=0.5)
    ax.set_xlabel("Recall@10", fontsize=11)
    ax.set_ylabel("Index Size (MB)", fontsize=11)
    ax.set_title(_DATASET_LABELS[dataset], fontsize=13)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle("Index Memory vs Recall@10", fontsize=15, y=1.01)
fig.tight_layout()
_save_poster_fig(fig, "memory_vs_recall.png")

Saved -> results\figures\memory_vs_recall.png


In [ ]:
# --- Figure 3 : Parameter Sweep Curves ---------------------------------------

_PARAM_NAME = {"IVFPQ": "nprobe", "HNSWSQ": "efSearch", "LSH": "nbits"}

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for col_i, method in enumerate(METHOD_ORDER):
    shades = _FACTORY_SHADES[method]
    param  = _PARAM_NAME[method]

    for row_i, dataset in enumerate(_DATASETS):
        ax  = axes[row_i][col_i]
        ax2 = ax.twinx()
        m_df = sweep_df[
            (sweep_df["method"] == method) & (sweep_df["dataset"] == dataset)
        ]

        for fi, (factory, grp) in enumerate(sorted(m_df.groupby("factory"))):
            grp   = grp.sort_values("search_param_value")
            color = shades[fi % len(shades)]
            short = factory.split(",", 1)[-1] if "," in factory else factory

            if method == "LSH":
                # nbits is structural (each value = different index) -> discrete scatter
                ax.scatter(grp["search_param_value"], grp["recall_at_10"],
                           color=color, s=55, zorder=3, label=short)
                ax2.scatter(grp["search_param_value"], grp["qps"],
                            color=color, marker="s", s=35, alpha=0.45, zorder=2)
            else:
                ax.plot(grp["search_param_value"], grp["recall_at_10"],
                        marker="o", color=color, linewidth=1.8, markersize=4,
                        label=short)
                ax2.plot(grp["search_param_value"], grp["qps"],
                         color=color, linewidth=1.0, linestyle="--",
                         alpha=0.45, marker="s", markersize=3)

        ax.set_xlabel(param, fontsize=9)
        ax.set_ylabel("Recall@10", fontsize=9)
        ax2.set_ylabel("QPS", fontsize=8, color="#888888")
        ax2.tick_params(axis="y", labelcolor="#888888", labelsize=7)
        ax.set_title(f"{_DATASET_LABELS[dataset]} - {method}", fontsize=10)
        ax.grid(alpha=0.25)
        ax.legend(fontsize=7, loc="lower right")
        xticks = sorted(m_df["search_param_value"].unique())
        ax.set_xticks(xticks)
        ax.set_xticklabels([str(v) for v in xticks], fontsize=7)

fig.suptitle(
    "Parameter Sweep  -  Recall@10 (solid) & QPS (dashed) vs Tuning Parameter",
    fontsize=13,
)
fig.tight_layout()
_save_poster_fig(fig, "param_sweeps.png")

Saved -> results\figures\param_sweeps.png


In [ ]:
# --- Figure : param_sweeps_clean.png -----------------------------------------

import matplotlib.ticker as _mticker
import matplotlib as mpl
mpl.rcParams.update({"font.family": "serif"})

# -- Load sweep data (memory-first, then results.csv, then tables csv) ---------
try:
    _df = sweep_df.copy()
except NameError:
    for _p in [
        PROJECT_ROOT / "results" / "results.csv",
        PROJECT_ROOT / "results" / "tables" / "all_datasets_results.csv",
    ]:
        if _p.exists():
            _df = pd.read_csv(_p)
            break
    else:
        raise FileNotFoundError("sweep data not found -- run cell 28 or the experiment cells first")
    if "index_size_mb" in _df.columns:
        _df = _df.rename(columns={"index_size_mb": "memory_mb"})

# -- Fixed structural config per method/dataset --------------------------------
_IVFPQ_FACTORY = {
    "sift":  "IVF4096,PQ32x8",
    "gist":  "IVF4096,PQ32x8",
    "glove": "IVF4096,PQ25x8",
}
_HNSWSQ_FACTORY = "HNSW32_SQ8"

ivfpq_df = pd.concat(
    [_df[(_df["dataset"] == ds) & (_df["method"] == "IVFPQ") & (_df["factory"] == fac)]
     for ds, fac in _IVFPQ_FACTORY.items()],
    ignore_index=True,
)
hnswsq_df = _df[(_df["method"] == "HNSWSQ") & (_df["factory"] == _HNSWSQ_FACTORY)].copy()
lsh_df    = _df[_df["method"] == "LSH"].copy()

print(f"Rows used  -- IVFPQ: {len(ivfpq_df)}, HNSWSQ: {len(hnswsq_df)}, LSH: {len(lsh_df)}")
print(f"Configs    -- {_IVFPQ_FACTORY}  |  HNSWSQ: {_HNSWSQ_FACTORY}  |  LSH: IndexLSH (all nbits)")

# -- Visual style --------------------------------------------------------------
_DS_COLOR  = {"sift": "#2D5A27", "gist": "#8B7D3A", "glove": "#8F9779"}
_DS_LABEL  = {"sift": "SIFT1M",  "gist": "GIST1M",  "glove": "GloVe-100d"}
_DS_MARKER = {"sift": "o",       "gist": "s",        "glove": "^"}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)

_panels = [
    ("IVFPQ",  ivfpq_df,  [1, 4, 8, 16, 32, 64],  "nprobe",   "log"),
    ("HNSWSQ", hnswsq_df, [16, 32, 64, 128, 256],  "efSearch", "linear"),
    ("LSH",    lsh_df,    [128, 256, 512],          "nbits",    "linear"),
]

for ax, (method, mdf, xticks, xlabel, xscale) in zip(axes, _panels):
    for ds in ["sift", "gist", "glove"]:
        sub = mdf[mdf["dataset"] == ds].sort_values("search_param_value")
        if sub.empty:
            continue
        ax.plot(sub["search_param_value"], sub["recall_at_10"],
                marker=_DS_MARKER[ds], color=_DS_COLOR[ds],
                label=_DS_LABEL[ds], linewidth=2, markersize=6)

    ax.set_xscale(xscale)
    ax.set_xticks(xticks)
    ax.set_xticklabels([str(v) for v in xticks], fontsize=9)
    ax.xaxis.set_minor_locator(_mticker.NullLocator())
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_title(method, fontsize=13)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3, which="major")
    ax.legend(fontsize=9, framealpha=0.8)

axes[0].set_ylabel("Recall@10", fontsize=11)

_caption = (
    "IVFPQ: PQ32x8 (SIFT/GIST), PQ25x8 (GloVe)  |  "
    "HNSWSQ: HNSW32_SQ8  |  LSH: IndexLSH"
)
fig.suptitle(f"Parameter Sweep  --  Recall@10 {_caption}", fontsize=12, y=1.02)
fig.tight_layout()

_out = PROJECT_ROOT / "results" / "figures" / "param_sweeps_clean.png"
_out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(_out, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved -> {_out.relative_to(PROJECT_ROOT)}")

Rows used  -- IVFPQ: 18, HNSWSQ: 15, LSH: 9
Configs    -- {'sift': 'IVF4096,PQ32x8', 'gist': 'IVF4096,PQ32x8', 'glove': 'IVF4096,PQ25x8'}  |  HNSWSQ: HNSW32_SQ8  |  LSH: IndexLSH (all nbits)
Saved -> results\figures\param_sweeps_clean.png


In [ ]:
# --- Figure : hard_query_recall_qps.png --------------------------------------
# Recall@10 bar chart for hard-query low vs high effort,
# annotated above each high-effort bar with the QPS throughput cost.
# All data already in adaptive_hard_query_df / adaptive_hard_query_results.csv.

import matplotlib as mpl
import matplotlib.ticker as _mticker
mpl.rcParams.update({"font.family": "serif"})

# -- Load data (memory-first, then CSV) ----------------------------------------
try:
    _adf = adaptive_hard_query_df.copy()
except NameError:
    _csv = PROJECT_ROOT / "results" / "tables" / "adaptive_hard_query_results.csv"
    if not _csv.exists():
        raise FileNotFoundError(f"adaptive_hard_query_results.csv not found at {_csv}")
    _adf = pd.read_csv(_csv)

# -- Pivot to wide: one row per (dataset, method) with low/high columns --------
_min_val = _adf.groupby(["dataset", "method"])["search_param_value"].transform("min")
_adf["effort"] = np.where(_adf["search_param_value"] == _min_val, "low", "high")

_pivot_r = _adf.pivot_table(index=["dataset","method"], columns="effort",
                             values="recall_hard").reset_index()
_pivot_q = _adf.pivot_table(index=["dataset","method"], columns="effort",
                             values="qps_hard").reset_index()
_wide = _pivot_r.merge(_pivot_q, on=["dataset","method"],
                       suffixes=("_recall", "_qps"))
_wide["label"] = _wide["dataset"].str.upper() + "-" + _wide["method"]
_wide = _wide.sort_values(["dataset", "method"]).reset_index(drop=True)

print("Using rows (no re-run):")
print(_adf[["dataset","method","search_param_value","recall_hard","qps_hard"]].to_string(index=False))

# -- QPS annotation helper -----------------------------------------------------
def _fmt_qps(v: float) -> str:
    if v >= 10_000:
        return f"{v/1000:.0f}k"
    if v >= 1_000:
        return f"{v/1000:.1f}k"
    return str(int(round(v)))

# -- Plot ----------------------------------------------------------------------
_LOW_COLOR  = "#8F9779"   # sage        -> low / light effort
_HIGH_COLOR = "#2D5A27"   # forest green -> high / heavy effort

fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(len(_wide))
width = 0.35

ax.bar(x - width/2, _wide["low_recall"],  width=width,
       color=_LOW_COLOR,  label="Low effort",  alpha=0.9)
bars_high = ax.bar(x + width/2, _wide["high_recall"], width=width,
                   color=_HIGH_COLOR, label="High effort", alpha=0.9)

# Annotate above each HIGH bar: "qps_low -> qps_high  (+XX% QPS)"
for bar, (_, row) in zip(bars_high, _wide.iterrows()):
    q_lo = float(row["low_qps"])
    q_hi = float(row["high_qps"])
    pct  = int(round((q_hi - q_lo) / q_lo * 100))
    ann  = f"{_fmt_qps(q_lo)}→{_fmt_qps(q_hi)} ({pct:+d}%QPS)"
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.012,
        ann,
        ha="center", va="bottom",
        fontsize=7.5, fontfamily="serif",
        color="#333333",
    )

ax.set_xticks(x)
ax.set_xticklabels(_wide["label"], rotation=25, ha="right", fontsize=10)
ax.set_ylabel("Recall@10 (hard queries)", fontsize=11)
ax.set_ylim(0, 1.14)
ax.yaxis.set_major_locator(_mticker.MultipleLocator(0.2))
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=10)

_param_note = "IVFPQ nprobe: 4 (low) / 64 (high)  |  HNSWSQ efSearch: 32 (low) / 256 (high)"
fig.suptitle(
    f"Hard-query Recall@10: Low vs High Search Effort\n{_param_note}",
    fontsize=12, y=1.01,
)
fig.tight_layout()

_out = PROJECT_ROOT / "results" / "figures" / "hard_query_recall_qps.png"
_out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(_out, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"\nSaved -> {_out.relative_to(PROJECT_ROOT)}")

Using rows (no re-run):
dataset method  search_param_value  recall_hard     qps_hard
   sift  IVFPQ                   4      0.40608 65772.511271
   sift  IVFPQ                  64      0.66184  9429.777579
   sift HNSWSQ                  32      0.87520 18856.425669
   sift HNSWSQ                 256      0.98384  4968.156107
   gist  IVFPQ                   4      0.13600  9901.656771
   gist  IVFPQ                  64      0.21400  4761.596390
   gist HNSWSQ                  32      0.56360  4278.418970
   gist HNSWSQ                 256      0.90080   937.765388
  glove  IVFPQ                   4      0.38968 91037.536527
  glove  IVFPQ                  64      0.61916 32518.675500
  glove HNSWSQ                  32      0.79776 16817.632950
  glove HNSWSQ                 256      0.95640  6115.534682

Saved -> results\figures\hard_query_recall_qps.png
